# Dofbot AI Controller — Simulation Notebook

This notebook simulates the full Dofbot AI Controller pipeline without any hardware.
You can explore how the system works step by step:

1. Load the movement library
2. See how servo angles are stored
3. Simulate running a movement
4. Call Groq AI and see what it picks
5. Visualize servo angles as a chart
6. Run multi-step sequences

**No hardware required. No Arm_Lib needed.**

## Step 1 — Load the movement library

In [ ]:
import json
import os

# Load movements.json from same folder as this notebook
with open('movements.json') as f:
    MOVEMENTS = json.load(f)

print(f'Loaded {len(MOVEMENTS)} movements:\n')
for name, mv in MOVEMENTS.items():
    print(f'  {name:<20} — {len(mv["steps"])} steps — {mv["description"]}')

## Step 2 — Inspect a movement's servo angles

In [ ]:
# Change this to any movement name
MOVEMENT_NAME = 'pick up'

mv = MOVEMENTS[MOVEMENT_NAME]
print(f'Movement: {MOVEMENT_NAME}')
print(f'Description: {mv["description"]}')
print(f'Steps: {len(mv["steps"])}\n')
print(f'{"Step":<6} {"S1":>5} {"S2":>5} {"S3":>5} {"S4":>5} {"S5":>5} {"S6":>5} {"Speed":>7} {"Sleep":>6}')
print('-' * 55)
for i, step in enumerate(mv['steps']):
    s1,s2,s3,s4,s5,s6,speed,sleep = step
    print(f'{i+1:<6} {s1:>5} {s2:>5} {s3:>5} {s4:>5} {s5:>5} {s6:>5} {speed:>7} {sleep:>6}')

## Step 3 — Simulate running a movement

In [ ]:
import time

class SimulatedArm:
    """Simulated arm — prints commands instead of moving hardware."""
    def __init__(self):
        self.history = []  # records all positions

    def Arm_serial_set_torque(self, state):
        print(f'  [SIM] Torque {"ON" if state else "OFF"}')

    def Arm_serial_servo_write6(self, s1, s2, s3, s4, s5, s6, speed):
        print(f'  [SIM] MOVE  s1={s1:>3}  s2={s2:>3}  s3={s3:>3}  s4={s4:>3}  s5={s5:>3}  s6={s6:>3}  speed={speed}')
        self.history.append([s1, s2, s3, s4, s5, s6])

def run_movement(name, arm):
    if name not in MOVEMENTS:
        print(f'Unknown movement: {name}')
        return
    print(f'\nRunning: {name}')
    print('-' * 40)
    arm.Arm_serial_set_torque(True)
    for step in MOVEMENTS[name]['steps']:
        s1,s2,s3,s4,s5,s6,speed,sleep = step
        arm.Arm_serial_servo_write6(s1,s2,s3,s4,s5,s6,speed)
    print('-' * 40)
    print('Done!')

# Try it
arm = SimulatedArm()
run_movement('yes', arm)

## Step 4 — Visualize servo angles as a chart

In [ ]:
# Install matplotlib if needed: pip install matplotlib
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d0d0d'
matplotlib.rcParams['axes.facecolor']   = '#111'
matplotlib.rcParams['axes.edgecolor']   = '#333'
matplotlib.rcParams['text.color']       = '#888'
matplotlib.rcParams['xtick.color']      = '#555'
matplotlib.rcParams['ytick.color']      = '#555'

# Run a movement and capture history
arm2 = SimulatedArm()
MOVEMENT_TO_PLOT = 'pick up'  # change this
run_movement(MOVEMENT_TO_PLOT, arm2)

if arm2.history:
    history = arm2.history
    steps   = list(range(1, len(history)+1))
    labels  = ['S1 base','S2 shoulder','S3 elbow','S4 wrist','S5 rotate','S6 gripper']
    colors  = ['#2a6','#3b7','#4c8','#5d9','#6ea','#7fb']

    fig, ax = plt.subplots(figsize=(10, 5))
    for i in range(6):
        vals = [h[i] for h in history]
        ax.plot(steps, vals, marker='o', label=labels[i], color=colors[i], linewidth=2)

    ax.set_title(f'Servo Angles — {MOVEMENT_TO_PLOT}', color='#aaa', pad=12)
    ax.set_xlabel('Step', color='#555')
    ax.set_ylabel('Angle (degrees)', color='#555')
    ax.set_ylim(-5, 185)
    ax.legend(loc='upper right', framealpha=0.2, labelcolor='linecolor')
    ax.grid(True, color='#1a1a1a', linewidth=0.8)
    plt.tight_layout()
    plt.show()
else:
    print('No history to plot.')

## Step 5 — Run a multi-step sequence

In [ ]:
def run_sequence(names):
    """Run multiple movements in order, like the real API does."""
    arm = SimulatedArm()
    holding = False
    print(f'Sequence: {" → ".join(names)}\n')
    for name in names:
        run_movement(name, arm)
        if 'pick up' in name: holding = True
        if 'put'     in name: holding = False
    print(f'\nTotal steps executed: {len(arm.history)}')
    print(f'Final servo positions: {arm.history[-1] if arm.history else "none"}')
    return arm

# Try: pick up from front then put it to the left
arm3 = run_sequence(['pick up', 'put left'])

## Step 6 — Call Groq AI (requires API key)

This shows how Groq picks movements from the library.

In [ ]:
import requests as req

GROQ_KEY = 'your_groq_api_key_here'  # replace with yours from console.groq.com

def ask_groq(instruction):
    mv_list = '\n'.join(f'- "{n}": {m["description"]}' for n,m in MOVEMENTS.items())
    prompt  = f"""Pick movements from this list and return ONLY a JSON array.

MOVEMENTS:\n{mv_list}

RULES:
- Return ONLY a JSON array like [\"pick up\",\"home\"]
- NEVER add \"home\" after pick up
- Only release with put when user says put/place/drop
- For look/celebrate/yes/no: end with \"home\"

User: {instruction}"""

    r = req.post('https://api.groq.com/openai/v1/chat/completions',
        json={'model':'llama-3.1-8b-instant',
              'messages':[{'role':'user','content':prompt}],
              'temperature':0.1,'max_tokens':200},
        headers={'Authorization':'Bearer '+GROQ_KEY,'Content-Type':'application/json'},
        timeout=30)
    raw   = r.json()['choices'][0]['message']['content'].strip()
    start = raw.find('['); end = raw.rfind(']')+1
    return json.loads(raw[start:end])

if GROQ_KEY != 'your_groq_api_key_here':
    # Test some instructions
    tests = [
        'pick up from front',
        'pick up and put it to the left',
        'say yes',
        'look left then right',
        'celebrate',
    ]
    print(f'{"Instruction":<35} -> Movements')
    print('-' * 65)
    for t in tests:
        mvs = ask_groq(t)
        print(f'{t:<35} -> {mvs}')
else:
    print('Add your Groq API key above to test AI movement selection.')
    print('Get one free at https://console.groq.com')

## Step 7 — Compare all movements visually

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show final servo position of each movement as a bar chart
names   = list(MOVEMENTS.keys())
labels  = ['S1','S2','S3','S4','S5','S6']
colors  = ['#2a6','#3b7','#4c8','#5d9','#6ea','#7fb']

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('All Movements — Final Servo Positions', color='#888', fontsize=13, y=0.98)

for idx, name in enumerate(names[:18]):
    ax  = axes[idx // 6][idx % 6]
    mv  = MOVEMENTS[name]
    pos = mv['steps'][-1][:6]  # last step servo values

    ax.set_facecolor('#111')
    bars = ax.bar(labels, pos, color=colors, alpha=0.8, width=0.6)
    ax.set_ylim(0, 190)
    ax.set_title(name, color='#aaa', fontsize=7, pad=4)
    ax.tick_params(colors='#444', labelsize=6)
    ax.spines[:].set_color('#222')
    ax.axhline(90, color='#333', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()